# UR5 Reach Design Notes: Main Class And Warp

这份笔记专门讲代码设计思路，也会明确指到对应代码。
这次会把**奖励机制设计**作为重点来讲，因为实际调训练时，最常动的就是 reward。

## 1. 整体分层

两条训练线都遵守同一套高层分层：

1. `*_config.py`：定义环境参数、训练参数、产物路径。
2. `*_env.py`：定义任务环境、观测、奖励、终止条件。
3. `train_*.py`：负责 CLI、训练循环、模型保存、日志和最终评估。

对应文件：
- Main class 线：`ur5_reach_config.py`、`ur5_reach_env.py`、`train_ur5_reach.py`
- Warp 线：`warp_ur5_config.py`、`warp_ur5_env.py`、`train_ur5_reach_warp.py`

为什么这样拆：
- 调参数默认值时只碰 config。
- 改 reward 和 done 逻辑时只碰 env。
- 改训练入口、日志和模型目录时只碰 train 脚本。

## 2. Main Class 线

主线是 `MuJoCo + Gymnasium + Stable-Baselines3`。
建议阅读顺序：

1. `ur5_reach_config.py`
2. `ur5_reach_env.py`
3. `train_ur5_reach.py`

### 2.1 配置层对应的代码

主线配置文件关键入口：
- `UR5ReachEnvConfig`：`ur5_reach_config.py:133`
- `RLTrainConfig`：`ur5_reach_config.py:269`
- `build_run_paths(...)`：`ur5_reach_config.py:375`

这三块分别负责：
- 环境与奖励默认值
- 算法训练默认值
- 模型与日志目录统一规则

重要设计点：
- CLI 默认值来自 dataclass
- 文档里的参数解释和代码配置尽量同源
- `best_model` / `final_model` 目录规则统一收在 config 层，而不是散在训练脚本里

In [ ]:
from ur5_reach_config import UR5ReachEnvConfig, RLTrainConfig, build_run_paths

env_cfg = UR5ReachEnvConfig()
train_cfg = RLTrainConfig()
paths = build_run_paths('sac', 'demo_run')

print('control_mode:', env_cfg.control_mode)
print('step_penalty:', env_cfg.step_penalty)
print('success_bonus:', env_cfg.success_bonus)
print('train algo:', train_cfg.algo)
print('run dir:', paths.run_dir)

### 2.2 环境 class 的关键入口在哪

主线环境重要函数位置：
- `class UR5ReachEnv`：`ur5_reach_env.py:37`
- `_finger_center(...)`：`ur5_reach_env.py:194`
- `_action_to_control(...)`：`ur5_reach_env.py:286`
- `_compute_reward(...)`：`ur5_reach_env.py:325`
- `reset(...)`：`ur5_reach_env.py:475`
- `step(...)`：`ur5_reach_env.py:538`

推荐阅读顺序：

1. `__init__`：模型、关节索引、观测空间、动作空间。
2. `_finger_center`：为什么成功参考点是两指中点。
3. `_action_to_control`：策略输出怎样变成控制量。
4. `_compute_reward`：奖励设计核心。
5. `reset` / `step`：环境状态如何推进。

In [ ]:
from ur5_reach_env import UR5ReachEnv

env = UR5ReachEnv(render_mode=None)
obs, info = env.reset()
print('obs shape:', obs.shape)
print('observation schema:')
for item in env.observation_schema():
    print(item)
env.close()

### 2.3 主线奖励机制设计：整体哲学

主线 reward 设计集中在 `ur5_reach_env.py:_compute_reward(...)`，代码位置是 `ur5_reach_env.py:325`。

主线的奖励设计不是“只看成功与否”，而是四层叠起来：

1. **基础时间与距离成本**
- `step_penalty`
- `distance_penalty`

2. **逼近过程 shaping**
- `progress_reward`
- `regress_penalty`
- `phase_reward`
- `direction_reward`

3. **稳定性与控制代价**
- `speed_penalty`
- `action_penalty`
- `smoothness_penalty`
- `joint_velocity_penalty`
- wrist 专项奖惩

4. **事件型奖励与终止**
- `collision_penalty`
- `success_bonus`
- `runaway_penalty`

这四层叠加的设计意图是：
- 不让策略只靠最终成功学东西
- 让“靠近目标的过程”本身就有反馈
- 同时明确压制抖动、乱转和碰撞

### 2.4 主线奖励机制设计：每一类奖励为什么存在

**A. 基础时间与距离成本**

- `step_penalty`
  - 作用：让策略知道“拖时间”本身有成本。
  - 如果没有它，策略可能学会一直抖、一直挪，但不着急完成。

- `distance_penalty`
  - 作用：让距离目标更远的状态本身更差。
  - 现在用 `sqrt(distance)` 而不是线性距离，是为了让远距离也能有明显梯度，但近距离不会陡得太离谱。

**B. 逼近过程 shaping**

- `progress_reward`
  - 作用：奖励“这一步比之前最好状态更近”。
  - 它会鼓励持续突破，而不是只在局部来回震荡。

- `regress_penalty`
  - 作用：如果比上一时刻更远了，就明确扣分。
  - 它和 `progress_reward` 组合起来，相当于让“向前走”和“往后退”有非对称代价。

- `phase_reward`
  - 作用：当距离第一次跨过若干阈值时，发一次性通关奖励。
  - 它的意义不是做最终精度，而是把“逼近过程”切成多个可学习的小阶段。

- `direction_reward`
  - 作用：如果末端实际运动方向和朝向目标的方向一致，就给奖励。
  - 它帮助策略学到“怎么朝目标动”，而不只是“最后距离变小了”。

**C. 稳定性与控制代价**

- `speed_penalty`
  - 作用：当末端速度过快时直接惩罚。
  - 这是抑制高速冲撞、末端抖动的第一道约束。

- `smoothness_penalty`
  - 作用：动作相对上一步变化太大就扣分。
  - 它压的是高频抖动和突然甩动。

- `joint_velocity_penalty`
  - 作用：关节速度变化太剧烈时扣分。
  - 它比 `smoothness_penalty` 更靠近真实关节状态，而不是只看策略输出。

**D. 事件型奖励与终止**

- `collision_penalty`
  - 作用：明确告诉策略，碰撞是严重错误。
  - 这类奖励通常做得很大，因为它不仅是回报问题，也是安全约束。

- `success_bonus`
  - 作用：让策略知道“真正完成任务”远比局部 shaping 更重要。
  - 配合剩余步数奖励与低速成功奖励，可以鼓励又快又稳地完成。

一句话概括：
- `distance/progress/phase/direction` 负责教策略怎么接近目标。
- `speed/smoothness/joint velocity/wrist` 负责教策略怎么稳定地接近目标。
- `collision/success` 负责告诉策略什么才算真正对或真正错。

### 2.5 主线奖励机制设计：为什么要保留 reward_terms 字典

主线没有把 reward 直接写成一个匿名标量，而是先放进 `reward_terms` 字典。

这样做有三个很实际的好处：

1. 测试模式可以直接打印每步奖励分解。
2. 调参时能知道到底是哪一类项在主导策略。
3. 新增 wrist 奖励或惩罚时，不容易把原有设计弄混。

所以 `reward_terms` 其实不只是调试辅助，它本身就是奖励机制设计可维护性的关键。

### 2.6 主线 wrist 奖惩为什么改成 gated reward

对应代码还是在 `ur5_reach_env.py:_compute_reward(...)` 的 wrist 段。

最近 wrist 的设计变化，本质上是在解决一个冲突：
- 如果完全不罚 wrist，模型会在末端附近高速空转。
- 如果把所有 wrist 旋转都视为坏事，又会误伤“接近目标时正常的小幅姿态微调”。

所以现在把 wrist 奖惩拆成两类：

**正常微调奖励**
- `wrist_alignment_reward`
- 只在接近目标、末端速度小、wrist 没有翻转、距离还在继续变好时才给。

**异常行为惩罚**
- `wrist_rotation_penalty`
- `wrist_action_smoothness_penalty`
- `wrist_speed_penalty`
- `wrist_direction_flip_penalty`

这个设计背后的奖励哲学是：
- 奖励“有用的小幅微调”
- 惩罚“无效的持续旋转、来回翻转、控制跳变”

这比“所有 wrist 旋转都罚”更像真实任务需求。

In [ ]:
from ur5_reach_config import UR5ReachEnvConfig

cfg = UR5ReachEnvConfig()
print('step_penalty:', cfg.step_penalty)
print('distance_weight:', cfg.distance_weight)
print('progress_reward_gain:', cfg.progress_reward_gain)
print('phase_thresholds:', cfg.phase_thresholds)
print('phase_rewards:', cfg.phase_rewards)
print('wrist micro threshold:', cfg.wrist_micro_adjustment_speed_threshold)
print('wrist alignment reward gain:', cfg.wrist_alignment_reward_gain)

### 2.7 主线训练脚本要怎么读

主线训练入口关键函数位置：
- `build_parser(...)`：`train_ur5_reach.py:328`
- `train(...)`：`train_ur5_reach.py:634`
- `resolve_test_paths(...)`：`train_ur5_reach.py:737`
- `evaluate_final_success_rate(...)`：`train_ur5_reach.py:800`
- `test(...)`：`train_ur5_reach.py:863`

这里要特别注意两件事：
- 测试命令不再手写模型路径，而是由 `resolve_test_paths(...)` 统一解析。
- 最终评估和训练日志都在训练脚本里负责串起来，而不是塞进环境类里。

In [ ]:
from train_ur5_reach import build_parser

parser = build_parser()
args = parser.parse_args(['--algo', 'sac', '--test', '--model', 'best'])
print(args)

## 3. Warp 线

Warp 线后端是 `Brax + MJX/Warp`。
建议阅读顺序：

1. `warp_ur5_config.py`
2. `warp_ur5_runtime.py`
3. `warp_ur5_env.py`
4. `train_ur5_reach_warp.py`

### 3.1 Warp 配置层对应的代码

Warp 配置文件关键入口：
- `WarpUR5EnvConfig`：`warp_ur5_config.py:76`
- `WarpTrainConfig`：`warp_ur5_config.py:169`
- `build_warp_run_paths(...)`：`warp_ur5_config.py:255`

它和主线保持统一风格，但不强行把 Brax 参数塞进 SB3 结构里。
这是一种“接口统一、实现分离”的设计。

In [ ]:
from warp_ur5_config import WarpUR5EnvConfig, WarpTrainConfig, build_warp_run_paths

env_cfg = WarpUR5EnvConfig()
train_cfg = WarpTrainConfig()
paths = build_warp_run_paths('sac', 'warp_demo')

print('reward mode:', env_cfg.reward_mode)
print('num_envs:', train_cfg.num_envs)
print('run dir:', paths.run_dir)

### 3.2 Warp 奖励机制设计：和主线的关系

Warp 奖励代码主要在 `warp_ur5_env.py` 的 `step(...)` 里，当前密集逻辑大约在 `warp_ur5_env.py:606` 到 `warp_ur5_env.py:653`。

Warp 的奖励哲学和主线是一致的，只是实现方式换成了 JAX 风格：

- `improvement_gain`：对应主线的 `progress_reward`
- `regress_gain`：对应主线的 `regress_penalty`
- `phase_rewards`：对应主线的阶段奖励
- `direction_reward_gain`：对应主线方向奖励
- `collision_penalty_value`：对应主线碰撞惩罚
- `success_bonus`：对应主线成功奖励

也就是说，两条线虽然后端不同，但奖励设计思想尽量保持一致：
- 靠近目标要有 shaping
- 控制太乱要有代价
- 碰撞要重罚
- 成功必须显著更值钱

### 3.3 Warp 奖励机制设计：为什么更偏训练工程而不是本地调试

主线 reward_terms 设计很适合本地一条条看。
Warp 线则更偏大规模训练工程，原因是：
- 它的环境和训练器是 JAX/Brax 风格
- 重点是并行吞吐和评估汇总，而不是本地 viewer 单步调试

所以 Warp 线更依赖：
- 聚合指标
- 训练回调日志
- 最终评估摘要

这不是说 Warp 不看 reward，而是 reward 的可视化方式和主线不同。

### 3.4 Warp 训练入口为什么只保留本地训练

Warp 训练脚本关键函数位置：
- `_build_final_eval_from_metrics(...)`：`train_ur5_reach_warp.py:140`
- `build_parser(...)`：`train_ur5_reach_warp.py:175`
- `train(...)`：`train_ur5_reach_warp.py:349`

这版设计的工程决策是：
- 本地 Windows 不强做 Warp test
- 本地先保证 Warp train + config + artifact layout 自洽
- 真正验证留给服务器环境

这样避免留下一个看起来完整、其实本地跑不通的假测试入口。

### 3.5 Warp 为什么用训练评估流产出 final_eval

主线 final_eval 是显式新开环境跑测试。
Warp 线当前采取不同策略：
- 直接复用 Brax 训练时的 eval metrics
- 汇总成 `success_rate`、`successes`、`min_distance`、`max_return`

代码入口是 `train_ur5_reach_warp.py:_build_final_eval_from_metrics(...)`。

这样做的工程价值是：
- 不依赖本地另外维护一套 Warp 推理脚本
- 训练结束就能稳定落一份 `final_eval.json`
- 更适合服务器无头训练

### 3.6 Warp 为什么 best_model 目前镜像 final_model

主线能直接保存 best model，是因为 SB3 的 `EvalCallback` 支持得比较直接。
Warp 当前这条 Brax 训练链稳定拿到的是最终参数，所以现在的折中是：
- 先统一目录契约：`best_model/`、`final_model/`
- 当前 best 先镜像 final export

这样后续如果服务器稳定拿到真正的 best params，只需要改导出逻辑，不需要重做整个目录结构。

## 4. 为什么统一目录规则

Main：
- `runs/{scope}/main/{algo}/{run_name}/...`

Warp：
- `runs/{scope}/warp/{algo}/{run_name}/...`

统一规则的直接收益：
- 测试命令只关心 `algo + run_name + model_variant`
- 本地和服务器只切换 `scope`
- 后面批量训练、批量评估、自动上传脚本可以复用一套思路

## 5. 推荐阅读顺序

如果你主要想看奖励机制：
1. 先读这份 notebook 的第 2.3、2.4、2.5、2.6 节
2. 再跳到 `ur5_reach_env.py:_compute_reward(...)`
3. 最后对照 `ur5_reach_config.py` 里的默认参数

如果你想读 Warp 奖励：
1. 先看这份 notebook 的第 3.2、3.3 节
2. 再跳到 `warp_ur5_env.py` 里 dense reward 那段逻辑
3. 最后看 `warp_ur5_config.py` 的对应系数

如果你想看完整工程：
1. `ur5_reach_config.py`
2. `ur5_reach_env.py`
3. `train_ur5_reach.py`
4. `warp_ur5_config.py`
5. `warp_ur5_env.py`
6. `train_ur5_reach_warp.py`